# OpenQSim — Colab sweep runner (high band 18-28q)

Runs a Phase 0A benchmark sweep on Colab, CPU-pinned, with checkpoints on
Google Drive so a disconnected session **resumes instead of restarting**.

**Run order:** mount Drive -> install -> clone -> run -> verify. Re-running
the same config in a new session skips already-done combos (Drive checkpoint).

**Why CPU:** Aer's GPU kernels don't run on Colab/Kaggle T4
(`cudaErrorNoKernelImageForDevice`); CPU is the consistent, working path and
matches the small-band data. Pinned to qiskit==1.4.2 / qiskit-aer==0.15.1
(newer Aer segfaults on qft/qaoa; qft.py needs the removed QFT class).

In [ ]:
# 1. Install pinned deps (CPU Aer = stable; GPU build is unusable on T4)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q',
                'qiskit', 'qiskit-aer', 'qiskit-aer-gpu', 'qiskit-terra'], check=False)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'qiskit==1.4.2', 'qiskit-aer==0.15.1',
    'pyyaml', 'networkx', 'pynvml', 'nvidia-ml-py'])
import qiskit, qiskit_aer
print('qiskit', qiskit.__version__, '| aer', qiskit_aer.__version__)

In [ ]:
# 2. Mount Drive -> checkpoints + results persist across sessions (resume)
from google.colab import drive
drive.mount('/content/drive')
import os
OUTPUT_DIR = '/content/drive/MyDrive/openqsim/benchmark_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('output ->', OUTPUT_DIR)

In [ ]:
# 3. Clone (or update) the repo on main
import os, subprocess
REPO = '/content/opensim-ai'
if os.path.isdir(REPO + '/.git'):
    subprocess.check_call(['git', '-C', REPO, 'pull', '--ff-only'])
else:
    subprocess.check_call(['git', 'clone', '--depth', '1',
                           'https://github.com/lekkalaharsha/opensim-ai', REPO])
print('repo ready:', REPO)

In [ ]:
# 4. Run the sweep (CPU, output+checkpoints on Drive, resumes automatically)
#    Pick the band. Checkpoint skips combos already done in OUTPUT_DIR.
#      sweep_config_colab.yaml        -> 18-28q, all 4 circuits (720)
#      sweep_config_colab_22-28.yaml  -> 22,24,26,28q only (480)
SWEEP = 'sweep_config_colab.yaml'
import os, subprocess, sys
env = dict(os.environ, OPENQSIM_DEVICE='CPU')  # T4 GPU kernels unusable -> CPU
subprocess.check_call([sys.executable, 'scripts/run_sweep.py',
                       '--config', f'benchmark/{SWEEP}',
                       '--output-dir', OUTPUT_DIR, '--no-push'],
                      cwd=REPO, env=env)

In [ ]:
# 5. Verify what's present vs the config (lists any missing combos)
import subprocess, sys
subprocess.run([sys.executable, 'scripts/verify_sweep.py',
                '--config', f'benchmark/{SWEEP}',
                '--raw', OUTPUT_DIR + '/raw'], cwd=REPO)

In [ ]:
# 6. (optional) Zip results for download off Drive
import shutil
z = shutil.make_archive('/content/drive/MyDrive/openqsim/benchmark_outputs_backup',
                        'zip', OUTPUT_DIR)
print('zip ->', z)